# Easy GRPO Kaggle Run

This notebook runs the shaped/easy GRPO curriculum for OnCallEnv Red Shift. It is separate from the hard Qwen2.5 3B GRPO notebook so the high-score easy result stays clearly labeled.

## 1. GPU Check

Expected on Kaggle: Tesla T4 GPUs. If this does not show GPUs, the notebook is not attached to the Kaggle GPU kernel.

In [ ]:
!nvidia-smi

Sun Apr 26 03:20:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Bootstrap Repo

The Kaggle kernel cannot see the local laptop path. This clones or updates the pushed `round2-redshift` branch into `/kaggle/working`.

In [6]:
import os
from pathlib import Path

Path('/kaggle/working').mkdir(parents=True, exist_ok=True)
os.chdir('/kaggle/working')
print('cwd:', os.getcwd())


cwd: /kaggle/working


In [7]:
import os, shutil, subprocess, time
from pathlib import Path

REPO_URL = 'https://github.com/srimanreddy4/MetaHackathon-R2'
BRANCH = 'round2-redshift'
WORKDIR = Path('/kaggle/working/MetaHackathon-R2')

os.chdir('/kaggle/working')
if (WORKDIR / '.git').exists():
    os.chdir(WORKDIR)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    if WORKDIR.exists():
        backup = WORKDIR.with_name(f'{WORKDIR.name}.bak.{int(time.time())}')
        shutil.move(str(WORKDIR), str(backup))
        print('Moved non-git existing directory to', backup)
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(WORKDIR)], check=True)
    os.chdir(WORKDIR)

print('cwd:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-5'], check=True)


Cloning into '/kaggle/working/MetaHackathon-R2'...


cwd: /kaggle/working/MetaHackathon-R2
49f394e eval: add rcaeval qwen transfer test
41bc706 docs: remove react checkpoint from readme
2cf64ee docs: report react checkpoint rewards
72de4f9 eval: plot react checkpoint rewards
a8a82c3 docs: report react sft partial curve


CompletedProcess(args=['git', 'log', '--oneline', '-5'], returncode=0)

In [8]:
%cd /kaggle/working/MetaHackathon-R2
%env PYTHONPATH=src:scripts
!python - <<'PY'
import os
from pathlib import Path
print('cwd=', Path.cwd())
print('PYTHONPATH=', os.environ.get('PYTHONPATH'))
print('repo exists=', Path('src/oncallenv').exists())



/kaggle/working
env: PYTHONPATH=src:scripts
/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
cwd= /kaggle/working/MetaHackathon-R2
PYTHONPATH= src:scripts
repo exists= True


## 3. Install Dependencies

Run once per Kaggle session.

In [9]:
!python -m pip install -U pip setuptools wheel
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-llm.txt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.3 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 18.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [openenv-core] [fastmcp]]ydantic]
INFO: pip is looking at multiple versions of unsloth to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of unsloth to dete

In [ ]:
# Fallback if dependency resolution fails:
# !python -m pip install -U transformers datasets accelerate trl peft bitsandbytes unsloth


## 4. Verify

Expected: `21 passed` and OpenEnv validation OK.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh verify

## 5. Easy Smoke Run

This validates easy prompts and shaped reward before spending more time. Expected score should be much higher than hard mode.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-smoke

In [ ]:
!cat training_results/unsloth_grpo_qwen3b_easy_smoke/summary.json

## 6. Easy Main Run

This is the high-score shaped-curriculum run: easy prompts, dense partial-credit reward, 300 steps.

In [9]:
!git pull
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-main

remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 12 (delta 10), reused 12 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 2.93 KiB | 428.00 KiB/s, done.
From https://github.com/srimanreddy4/MetaHackathon-R2
   82ae6b3..5affb0b  round2-redshift -> origin/round2-redshift
   4f7b330..2adf149  spicy-attacker  -> origin/spicy-attacker
Updating 82ae6b3..5affb0b
Fast-forward
 scripts/run_kaggle_qwen3b_grpo.sh |  4 ++--
 scripts/train_unsloth_grpo.py     | 14 +++++++++-----
 2 files changed, 11 insertions(+), 7 deletions(-)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 12:20:38.970520: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777119638.993934     869 cuda_dnn.cc:8579] Unable to register cuD

In [ ]:
!cat training_results/unsloth_grpo_qwen3b_easy/summary.json
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-summary

In [10]:
!git pull
!bash scripts/run_kaggle_qwen3b_grpo.sh export-easy-artifacts


remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 134 (delta 98), reused 93 (delta 58), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 248.03 KiB | 4.43 MiB/s, done.
Resolving deltas: 100% (98/98), completed with 22 local objects.
From https://github.com/srimanreddy4/MetaHackathon-R2
   5affb0b..93deb8c  round2-redshift -> origin/round2-redshift
   2adf149..4562fa7  spicy-attacker  -> origin/spicy-attacker
 * [new branch]      sre-three-way   -> origin/sre-three-way
Updating 5affb0b..93deb8c
Fast-forward
 .gitignore                                         |   3 +
 README.md                                          |  64 ++
 artifacts/README.md                                |  52 ++
 artifacts/models/.gitkeep                          |   0
 demo_episode.py                                    | 196 ++++++
 docs/EASY_EVAL_KT.md                               | 159 +++++
 docs/EAS

In [11]:
!bash scripts/run_kaggle_qwen3b_grpo.sh easy-eval-checkpoint
!bash scripts/run_kaggle_qwen3b_grpo.sh export-easy-artifacts


2026-04-25 17:29:05.735894: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777138145.758352    1083 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777138145.765888    1083 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777138145.786488    1083 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777138145.786515    1083 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777138145.786530    1083 computation_placer.cc:177] computation placer alr

In [17]:
from pathlib import Path
import shutil
from IPython.display import FileLink, display

# The export script writes the downloadable archive to /kaggle/working, not inside the repo.
archive_path = Path('/kaggle/working/easy_grpo_qwen3b_artifacts.tar.gz')
repo_copy = Path('/kaggle/working/MetaHackathon-R2/artifacts/models/easy_grpo_qwen3b_artifacts.tar.gz')
extract_dir = Path('/kaggle/working/MetaHackathon-R2/artifacts/models/easy_grpo_qwen3b')

if not archive_path.exists():
    print(f"File not found: {archive_path}")
    print("Run this first: !bash scripts/run_kaggle_qwen3b_grpo.sh export-easy-artifacts")
else:
    size_mb = archive_path.stat().st_size / (1024 * 1024)
    print(f"Archive exists: {archive_path} ({size_mb:.1f} MB)")

    repo_copy.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive_path, repo_copy)
    print(f"Copied into repo artifact folder: {repo_copy}")

    extract_dir.mkdir(parents=True, exist_ok=True)
    !tar -xzf /kaggle/working/easy_grpo_qwen3b_artifacts.tar.gz -C /kaggle/working/MetaHackathon-R2/artifacts/models/easy_grpo_qwen3b
    print(f"Extracted under: {extract_dir}")

    print("Download link for Kaggle/Jupyter:")
    display(FileLink(str(archive_path)))

    print("Repo copy link:")
    display(FileLink(str(repo_copy)))


Archive exists: /kaggle/working/easy_grpo_qwen3b_artifacts.tar.gz (643.5 MB)
Copied into repo artifact folder: /kaggle/working/MetaHackathon-R2/artifacts/models/easy_grpo_qwen3b_artifacts.tar.gz
Extracted under: /kaggle/working/MetaHackathon-R2/artifacts/models/easy_grpo_qwen3b
Download link for Kaggle/Jupyter:


/kaggle/working/easy_grpo_qwen3b_artifacts.tar.gz

Repo copy link:


/kaggle/working/MetaHackathon-R2/artifacts/models/easy_grpo_qwen3b_artifacts.tar.gz

In [6]:
!git pull
!bash scripts/run_kaggle_qwen3b_grpo.sh react-generate

Already up to date.
{
  "out_dir": "training_results/react_sft_qwen3b",
  "dataset_path": "training_results/react_sft_qwen3b/react_trajectories.jsonl",
  "num_tasks": 120,
  "train_tasks": 100,
  "eval_tasks": 20,
  "num_rows": 927,
  "train_rows": 774,
  "eval_rows": 153,
  "mean_expert_reward": 0.9714729166666666
}


In [11]:
!git pull
!bash scripts/run_kaggle_qwen3b_grpo.sh react-sft-main

Already up to date.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-25 19:51:52.649615: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777146712.675980     967 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777146712.684599     967 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777146712.706645     967 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777146712.706706     967 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than 

In [12]:
!git pull
!bash scripts/run_kaggle_qwen3b_grpo.sh react-eval-checkpoint


remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 33 (delta 23), reused 33 (delta 23), pack-reused 0 (from 0)
Unpacking objects: 100% (33/33), 126.24 KiB | 1.71 MiB/s, done.
From https://github.com/srimanreddy4/MetaHackathon-R2
   5238d65..a8a82c3  round2-redshift -> origin/round2-redshift
   7fc73d4..9c8272c  accel-attack    -> origin/accel-attack
   79308ee..42ed351  sre-three-way   -> origin/sre-three-way
Updating 5238d65..a8a82c3
Fast-forward
 README.md                                          |  29 +++++
 docs/plots/react_sft_qwen3b_partial_grad_norm.png  | Bin 0 -> 45567 bytes
 docs/plots/react_sft_qwen3b_partial_loss_curve.png | Bin 0 -> 55940 bytes
 .../react_sft_qwen3b_partial_loss_summary.png      | Bin 0 -> 33816 bytes
 docs/react_sft_interrupted_report.json             | 121 ++++++++++++++++++++
 scripts/evaluate_react_sft_checkpoint.py           | 123 +++++++++++++++++++++
 sc

In [13]:
!git pull
!bash scripts/run_kaggle_qwen3b_grpo.sh react-eval-checkpoint-fast


remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.38 KiB | 709.00 KiB/s, done.
From https://github.com/srimanreddy4/MetaHackathon-R2
   a8a82c3..72de4f9  round2-redshift -> origin/round2-redshift
Updating a8a82c3..72de4f9
Fast-forward
 scripts/evaluate_react_sft_checkpoint.py | 48 ++++++++++++++++++++++++++++++++
 scripts/run_kaggle_qwen3b_grpo.sh        | 20 ++++++++++++-
 2 files changed, 67 insertions(+), 1 deletion(-)
2026-04-25 20:40:47.868324: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777149647.896763    1157 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777149647.907594    1157 cuda_blas.cc:140

In [15]:
!git pull
!export PYTHONPATH=src:scripts && bash scripts/run_kaggle_qwen3b_grpo.sh rcaeval-qwen-test


remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 539 bytes | 539.00 KiB/s, done.
From https://github.com/srimanreddy4/MetaHackathon-R2
   9c87857..4bc12f1  round2-redshift -> origin/round2-redshift
Updating 9c87857..4bc12f1
Fast-forward
 scripts/evaluate_rcaeval_qwen_adapter.py | 16 ++++++++++++++--
 1 file changed, 14 insertions(+), 2 deletions(-)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
2026-04-26 04:25:07.361260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777177507.555460     231 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777177507.608974     231 cuda_blas.cc:1407] 

In [14]:
!find /kaggle/input -name '*easy*grpo*qwen3b*' -o -name '*.tar.gz' | sort


/kaggle/input/datasets/srimanreddy/artifacts/docs/plots/easy_grpo_qwen3b_checkpoint_results.png
/kaggle/input/datasets/srimanreddy/artifacts/docs/plots/easy_grpo_qwen3b_completion_health.png
/kaggle/input/datasets/srimanreddy/artifacts/docs/plots/easy_grpo_qwen3b_reward_curve.png
/kaggle/input/datasets/srimanreddy/artifacts/kaggle/working/easy_grpo_qwen3b_artifact_manifest.txt


## 7. Optional Fallback

Use this only if the 3B easy run OOMs or behaves badly.

In [ ]:
# MODEL_NAME=unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit bash scripts/run_kaggle_qwen3b_grpo.sh easy-main

## 8. Archive Outputs

Download `/kaggle/working/qwen3b_grpo_results.tar.gz` from Kaggle outputs.

In [ ]:
!bash scripts/run_kaggle_qwen3b_grpo.sh archive

## RCAEval public transfer test with saved Qwen adapter

Run this after the easy GRPO checkpoint exists. It loads `checkpoint-200` or the latest checkpoint, generates one RCAEval public-sample answer, scores it, and writes `eval_results/rcaeval_qwen_easy/summary.json`.


In [ ]:
!export PYTHONPATH=src:scripts && bash scripts/run_kaggle_qwen3b_grpo.sh rcaeval-qwen-test


In [ ]:
import json
from pathlib import Path
summary_path = Path('eval_results/rcaeval_qwen_easy/summary.json')
summary = json.loads(summary_path.read_text())
print(json.dumps({
    'checkpoint': summary.get('checkpoint'),
    'prediction': summary.get('prediction'),
    'score': summary.get('score'),
    'completion': summary.get('completion'),
    'plots': summary.get('plots'),
}, indent=2))
